In [12]:
import pandas as pd
import numpy as np
import json

### Load and preprocess the data ###
df = pd.read_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/OT_vs_OK_combined_annotations.csv")
                 
# Extract KEGG KOs and log2FoldChange
df = df[['KEGG_ko', 'log2FoldChange']]
df['KEGG_ko'] = df['KEGG_ko'].str.split(',')
df_exploded = df.explode('KEGG_ko')
df_exploded['KEGG_ko'] = df_exploded['KEGG_ko'].str.strip()
df_exploded = df_exploded.replace('-', np.nan).dropna()

with open("/home/nanopore/projects/rna_seq_workflow/resources/20250603_FCOM/enrichment_analysis/kegg_request/20250603_FCOM_kegg_cache.json", "r") as cache_file:
    kegg_cache = json.load(cache_file)

# Extract pathways and modules separately
df_exploded["Pathways"] = df_exploded["KEGG_ko"].map(lambda ko: kegg_cache.get(ko, {}).get("pathways", []))

df_pathways = df_exploded.explode("Pathways").dropna(subset=["Pathways"])

### Prepare data for GSEA ###
def prepare_gsea_data(df, term_column):
    """Prepare ranked list and gene sets for GSEA analysis."""
    ranked_list = (
        df.groupby("KEGG_ko")["log2FoldChange"]
        .mean()
        .sort_values(ascending=False)
    )
    gene_sets = (
        df.set_index(term_column)['KEGG_ko']
        .groupby(term_column)
        .apply(list)
        .to_dict()
    )
    return ranked_list, gene_sets

ranked_list_pathways, gene_sets_pathways = prepare_gsea_data(df_pathways, "Pathways")

ranked_list_pathways

KEGG_ko
ko:K15877    3.669337
ko:K09831    2.877755
ko:K00958    2.819685
ko:K00390    2.728562
ko:K00262    2.585547
               ...   
ko:K01279   -3.477900
ko:K00273   -3.646153
ko:K14541   -4.140986
ko:K00463   -4.318187
ko:K03787   -6.426687
Name: log2FoldChange, Length: 2060, dtype: float64